In [47]:
from pathlib import Path
import ast 
import pandas as pd

train_df = pd.read_csv(r"C:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE\data\raw\train_data.csv")
test_df = pd.read_csv(r"C:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE\data\raw\test_data.csv")

print("training conversations:", len(train_df))
print("testing conversations:", len(test_df))

training conversations: 450
testing conversations: 900


## Conversation identifiers

Each original conversation needs a permanent identifier before it is separated into message turns. The supplied training and testing origins are also retained for traceability. These source indicators are documentation fields and will not be used as model features.

In [48]:
train_working = train_df.copy()
test_working = test_df.copy()

train_working["source_split"] = "authors_train"
test_working["source_split"] = "authors_test"

train_working["source_row"] = range(len(train_working))
test_working["source_row"] = range(len(test_working))

train_working["conversation_id"] = [
    f"train_{row_number:04d}"
    for row_number in range(len(train_working))
]

test_working["conversation_id"] = [
    f"test_{row_number:04d}"
    for row_number in range(len(test_working))
]

working_df = pd.concat(
    [train_working, test_working], 
ignore_index=True
)

print("Combined conversations:", len(working_df)
    )

print(
    "Unique conversation IDs:", working_df["conversation_id"].nunique()
)

Combined conversations: 1350
Unique conversation IDs: 1350


In [49]:
print("Training working rows:", len(train_working))
print("Testing working rows:", len(test_working))
print("Combined working rows:", len(working_df))

print("\nFirst training IDs:")
print(train_working["conversation_id"].head().tolist())

print("\nFirst testing IDs:")
print(test_working["conversation_id"].head().tolist())

Training working rows: 450
Testing working rows: 900
Combined working rows: 1350

First training IDs:
['train_0000', 'train_0001', 'train_0002', 'train_0003', 'train_0004']

First testing IDs:
['test_0000', 'test_0001', 'test_0002', 'test_0003', 'test_0004']


## Parse Serialized Conversations
The original CSV stores each complete conveersation as text representing a python list. The converstion field needs to be parsed into a list before individual entries can be normalized and seperated into ordered turns. 

In [50]:
def parse_conversation(conversation_text): 
    """Convert a serialized conversation string into a python list."""

    conversation = ast.literal_eval(conversation_text)

    if not isinstance(conversation, list):
        raise ValueError("Conversation text did not parse into a list.")

    return conversation
working_df["parsed_conversation"] = working_df["conversation"].apply(parse_conversation)

print("Conversations parsed:", len(working_df)) 



Conversations parsed: 1350


In [51]:
working_df["number_of_entries"] = working_df["parsed_conversation"].apply(len)

print("Conversation entry counts:") 
display( working_df["number_of_entries"].value_counts().sort_index())

print("Non-list conversations:",
sum(not isinstance(conversation,list) 
    for conversation in working_df["parsed_conversation"]
                                   ))

Conversation entry counts:


number_of_entries
11    1350
Name: count, dtype: int64

Non-list conversations: 0


## Parsing Result 
 All 1350 serialized conversations were successfully organized into python lists. Every parsed conversation contains 11 ordered entries. The indivdiual entries still include a miscture od dictionaries, strogs, and nested list so they need to be normalized before turn level rows are created. 

In [52]:
import json
import re
import ast

In [53]:
# dictionaries and nested links 

def unwrap_entry(entry):
    """Unwrap entries that are already dictionaries or one-item lists."""
    if isinstance(entry, dict):
        return entry
    if isinstance(entry, list) and len(entry) == 1:
        return unwrap_entry(entry[0])
    
    return entry

In [54]:
#Test 

regular_entry = working_df.loc[
    0, "parsed_conversation"
][0]

unwrapped_entry = unwrap_entry(regular_entry)

print("Original type:", type(regular_entry).__name__)
print("Unwrapped type:", type(unwrapped_entry).__name__)
print("Fields:", unwrapped_entry.keys())

Original type: dict
Unwrapped type: dict
Fields: dict_keys(['Name', 'Message'])


In [55]:
#string entries parsing 

def parse_string_entry(entry):
    """Convert a serialized string entry into a dictionary."""
    text = entry.strip()

    for parser in (json.loads, ast.literal_eval):
        try:
            parsed_entry = parser(text)

            if isinstance(parsed_entry, dict):
                return parsed_entry
            if(
                isinstance(parsed_entry, list) 
                and len(parsed_entry) == 1
                and isinstance(parsed_entry[0], dict)
            ):
                return parsed_entry[0]
            
        except (ValueError, SyntaxError, json.JSONDecodeError):
            continue

    return None

In [56]:
# string test 

string_example = None
for conversation in working_df["parsed_conversation"]:
    for entry in conversation: 
        if isinstance(entry, str):
            string_example = entry
            break
    if string_example is not None:
        break

parsed_string_example = parse_string_entry(string_example)

print("Original type:", type(string_example).__name__) 
print("Parsed type:", type(parsed_string_example).__name__) 

if parsed_string_example is not None: 
    print("Fields:", parsed_string_example.keys())
            

Original type: str
Parsed type: dict
Fields: dict_keys(['Name', 'Message'])


In [57]:
# Normalize One Entry 

def normalize_entry(entry):
    """Convert an Entry into a standard Name and Message dictionary."""

    entry = unwrap_entry(entry)

    if isinstance(entry, str): 
        entry = parse_string_entry(entry)
    if not isinstance(entry, dict):
        return None

    if "Name" not in entry or "Message" not in entry:
        return None

    name = str(entry["Name"]).strip()
    message = str(entry["Message"]).strip()

    if not name or not message:
        return None

    return {
        "name": name,
        "message": message
    }

In [58]:
print(
  "regular entry:", 
  normalize_entry(regular_entry) is not None
 )

print(
 "string entry:",
 normalize_entry(string_example) is not None
)

regular entry: True
string entry: True


In [59]:
# Every Entry Test without Data Change 
successful_entries = 0
failed_entries = []

for _, row in working_df.iterrows():
    for turn_number, entry in enumerate(
        row["parsed_conversation"],
        start=1
    ):
        normalized = normalize_entry(entry)
        if normalized is None:
            failed_entries.append({
                "conversation_id": row["conversation_id"],
                "turn_number": turn_number,
                "entry_type": type(entry).__name__
                })
            
        else: 
            successful_entries += 1

print("Successfully Normalized Entries:", successful_entries)
print("Entries needing additional cleaning:", len(failed_entries))

            

Successfully Normalized Entries: 14662
Entries needing additional cleaning: 188


<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\i" is an i

## Initial Normalization

The intital Normalization process successfully standardized 14662 of the 14850 conversation entries. The 188 that were not standardized contain additional formatting irregularities and require further cleaning. All entries were accounted for with no record deletion. Repeated invalid escape warnings were from formatting in the original serialized conversation strings. 

In [60]:
#failure list table 

failed_entries_df = pd.DataFrame(failed_entries) 

print("Failed entries:", len(failed_entries_df)) 

print(
    "Conversations affected:", 
    failed_entries_df["conversation_id"].nunique()
)

display(failed_entries_df.head())

Failed entries: 188
Conversations affected: 77


,conversation_id,turn_number,entry_type
0,train_0057,6,str
1,train_0067,6,str
2,train_0084,8,str
3,train_0084,10,str
4,train_0118,5,str


In [61]:
#count failures by entry type
failure_type_counts = (
    failed_entries_df["entry_type"].value_counts()
    .value_counts()
    .rename_axis("entry type")
    .reset_index(name="failed entries")
)

display(failure_type_counts)

,entry type,failed entries
0,188,1


In [62]:
#count failures by source file 
failed_entries_df["source_split"] = (
    failed_entries_df["conversation_id"]
    .str.split("_")
    .str[0]
)

failed_source_counts = (
    failed_entries_df["source_split"].value_counts()
    .rename_axis("source file")
    .reset_index(name="failed entries")
)

display(failed_source_counts)

,source file,failed entries
0,test,141
1,train,47


In [63]:
#turns failure frequency 
failure_turn_counts = (
    failed_entries_df["turn_number"]
    .value_counts()
    .sort_index()
    .rename_axis("turn number")
    .reset_index(name="failed entries")
)

display(failure_turn_counts)

,turn number,failed entries
0,1,1
1,2,2
2,3,5
3,4,17
4,5,12
5,6,40
6,7,18
7,8,35
8,9,16
9,10,30


## Remaining Normalization Failures 

The initial parser accounted for 14850 entries but could not standardize 188 of them. Failures were analyzed by original entry type, source file, affected conversation and turn position. This will guid the next cleaning rule without deleting or manually altering the original data.

In [65]:
# additional cleaning analysis
failed_entries_df = pd.DataFrame(failed_entries)

print("Failed Entries:", len(failed_entries_df))

print(
    "conversations affected:",
    failed_entries_df["conversation_id"].nunique()
)

display(failed_entries_df.head())


Failed Entries: 188
conversations affected: 77


,conversation_id,turn_number,entry_type
0,train_0057,6,str
1,train_0067,6,str
2,train_0084,8,str
3,train_0084,10,str
4,train_0118,5,str


## Remaining Normalization Failures 

The initial normalization identified entries that could not be converted using the standard dictionary, nested-list, or serialized-string rules. Entries were retained and recorded by conversation id, turn number and original data type for further analysis. 

In [68]:
# failure entry type 
failure_type_counts = (
    failed_entries_df["entry_type"].value_counts()
   .rename_axis("entry_type")
   .reset_index(name="failed entries")
)

display(failure_type_counts)

,entry_type,failed entries
0,str,188


In [69]:
# failures from training or testing
failed_entries_df["source_file"] = (
    failed_entries_df["conversation_id"]
    .str.split("_")
    .str[0]
)

failure_source_counts = (
    failed_entries_df["source_file"].value_counts()
   .rename_axis("source_file")
   .reset_index(name="failed entries")
)

display(failure_source_counts)

,source_file,failed entries
0,test,141
1,train,47


### Failure Source 

The remaining normalization were grouped by their original source file. This determined whether the formatting problem is concetrated in the authors' training data, testing data, or distributed across both files 

### Failure position

The remaining failures were grouped by turn position to determine whether formatting problems occur throughout conversations or are concentrated at particular turns.

In [70]:
failure_turn_counts = (
    failed_entries_df["turn_number"]
    .value_counts()
    .sort_index()
    .rename_axis("turn_number")
    .reset_index(name="failed entries")
)

display(failure_turn_counts)

,turn_number,failed entries
0,1,1
1,2,2
2,3,5
3,4,17
4,5,12
5,6,40
6,7,18
7,8,35
8,9,16
9,10,30


### Failure summary

Most conversation entries were successfully normalized. The table below summarizes the remaining entries that require additional cleaning.

In [73]:
total_entries = 14850

failure_summary = pd.DataFrame({
    "Measure": [
        "Failed entries",
        "Affected Conversations",
        "Percentage of all entries"
    ],

    "Value" : [
        len(failed_entries_df),
        failed_entries_df["conversation_id"].nunique(),
        round(len(failed_entries_df) / total_entries * 100, 2)
    ]
})

display(failure_summary)

,Measure,Value
0,Failed entries,188.00
1,Affected Conversations,77.00
2,Percentage of all entries,1.27
